# Modeling / Architecture

This phase builds the first collaborative recommendation model.

## Objective

Use historical user-item ratings to learn personalized product
preference scores, which will later be used to rank candidate products.

## Model

Explicit Matrix Factorization implemented in PyTorch.

Ratings are centered around 3 stars:

- 1 → -1.0
- 2 → -0.5
- 3 → 0.0
- 4 → +0.5
- 5 → +1.0

The model learns separate latent representations for users and products.

## Important cold-start limitation

The collaborative model only learns item representations for products
with interactions in the training period (≤ 2021).

Products with zero training interactions remain in the global catalog
but cannot receive a collaborative score. This limitation is intentional
and will later be addressed using content-based representations.

In [2]:
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np


PROJECT_ROOT = Path.cwd().parent

INTERACTION_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "electronics_interactions"
    / "*.parquet"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

con = duckdb.connect()

In [3]:
users = con.sql(f"""
    SELECT DISTINCT user_id
    FROM read_parquet('{INTERACTION_PATH}')
    WHERE timestamp < 1640995200000
    ORDER BY user_id
""").df()

users["user_idx"] = np.arange(
    len(users),
    dtype=np.int32,
)

users = users[["user_idx", "user_id"]]

print(f"Training users: {len(users):,}")
users.head()

Training users: 16,058,864


,user_idx,user_id
0,0,AE2222C2STTMXNUHXTVDF55R7E3Q
1,1,AE2222FRPDMNOMYOMCWIANTXP7UQ
2,2,AE2222VGHFD37G7NOCWXASYY3ZDQ
3,3,AE2222WEGI5MUXLFZW6UKKUSI74A
4,4,AE22232OB6S7UAC75JUFYRGBSHIQ


In [4]:
mf_items = con.sql(f"""
    SELECT DISTINCT parent_asin
    FROM read_parquet('{INTERACTION_PATH}')
    WHERE timestamp < 1640995200000
    ORDER BY parent_asin
""").df()

mf_items["mf_item_idx"] = np.arange(
    len(mf_items),
    dtype=np.int32,
)

mf_items = mf_items[["mf_item_idx", "parent_asin"]]

print(f"Training items: {len(mf_items):,}")
mf_items.head()

Training items: 1,395,130


,mf_item_idx,parent_asin
0,0,0007182147
1,1,0011300000
2,2,0060009810
3,3,0060219602
4,4,0060897082


In [5]:
users.to_parquet(
    OUTPUT_DIR / "mf_users.parquet",
    index=False,
)

mf_items.to_parquet(
    OUTPUT_DIR / "mf_items.parquet",
    index=False,
)

In [6]:
assert users["user_idx"].is_unique
assert users["user_id"].is_unique

assert mf_items["mf_item_idx"].is_unique
assert mf_items["parent_asin"].is_unique

assert users["user_idx"].iloc[0] == 0
assert mf_items["mf_item_idx"].iloc[0] == 0

print("MF index mappings validated.")

MF index mappings validated.


In [7]:
rating_to_target = {
    1.0: -1.0,
    2.0: -0.5,
    3.0: 0.0,
    4.0: 0.5,
    5.0: 1.0,
}

In [8]:
train_interactions = con.sql(f"""
    SELECT
        u.user_idx,
        i.mf_item_idx,
        CASE rating
            WHEN 1.0 THEN -1.0
            WHEN 2.0 THEN -0.5
            WHEN 3.0 THEN  0.0
            WHEN 4.0 THEN  0.5
            WHEN 5.0 THEN  1.0
        END AS target
    FROM read_parquet('{INTERACTION_PATH}') r
    INNER JOIN users u
        ON r.user_id = u.user_id
    INNER JOIN mf_items i
        ON r.parent_asin = i.parent_asin
    WHERE EXTRACT(
        YEAR FROM to_timestamp(r.timestamp / 1000)
    ) <= 2021
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [9]:
print(train_interactions.shape)
print(train_interactions.head())
print(train_interactions.dtypes)
print(train_interactions["target"].value_counts().sort_index())

(37006469, 3)
   user_idx  mf_item_idx  target
0    819200      1384620     1.0
1    819201       334149     1.0
2    819202       967896     0.5
3    819203       277656     0.5
4    819204       569147     1.0
user_idx         int32
mf_item_idx      int32
target         float64
dtype: object
target
-1.0     4285643
-0.5     1864074
 0.0     2419023
 0.5     4840173
 1.0    23597556
Name: count, dtype: int64


### Preparing data

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader

user_indices = train_interactions["user_idx"].to_numpy(dtype=np.int64)
item_indices = train_interactions["mf_item_idx"].to_numpy(dtype=np.int64)
targets = train_interactions["target"].to_numpy(dtype=np.float32)

print(user_indices.nbytes / 1024**2, "MB")
print(item_indices.nbytes / 1024**2, "MB")
print(targets.nbytes / 1024**2, "MB")

282.33695220947266 MB
282.33695220947266 MB
141.16847610473633 MB


### MF Dataset

In [11]:
class MFDataset(Dataset):
    def __init__(self, user_indices, item_indices, targets):
        self.user_indices = user_indices
        self.item_indices = item_indices
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return (
            self.user_indices[idx],
            self.item_indices[idx],
            self.targets[idx],
        )

In [12]:
train_dataset = MFDataset(
    user_indices,
    item_indices,
    targets,
)

print("Training examples:", len(train_dataset))
print("First example:", train_dataset[0])

Training examples: 37006469
First example: (np.int64(819200), np.int64(1384620), np.float32(1.0))


### MF Model

In [13]:
import torch
import torch.nn as nn


LATENT_DIM = 16


class MatrixFactorization(nn.Module):
    def __init__(
        self,
        num_users,
        num_items,
        latent_dim=LATENT_DIM,
    ):
        super().__init__()

        self.user_embeddings = nn.Embedding(
            num_users,
            latent_dim,
            sparse=True,
        )

        self.item_embeddings = nn.Embedding(
            num_items,
            latent_dim,
            sparse=True,
        )

        self.user_bias = nn.Embedding(
            num_users,
            1,
            sparse=True,
        )

        self.item_bias = nn.Embedding(
            num_items,
            1,
            sparse=True,
        )

        self.global_bias = nn.Parameter(
            torch.tensor(0.0)
        )

    def forward(self, user_idx, item_idx):
        user_vec = self.user_embeddings(user_idx)
        item_vec = self.item_embeddings(item_idx)

        interaction = (user_vec * item_vec).sum(dim=1)

        user_bias = self.user_bias(user_idx).squeeze(-1)
        item_bias = self.item_bias(item_idx).squeeze(-1)

        return (
            interaction
            + user_bias
            + item_bias
            + self.global_bias
        )

In [14]:
num_users = len(users)
num_items = len(mf_items)

model = MatrixFactorization(
    num_users=num_users,
    num_items=num_items,
)

print(
    f"Users: {num_users:,}\n"
    f"Items: {num_items:,}\n"
    f"Latent dimensions: {LATENT_DIM}"
)

Users: 16,058,864
Items: 1,395,130
Latent dimensions: 16


In [16]:
TRAINING_QUERY = f"""
    SELECT
        u.user_idx,
        i.mf_item_idx,
        CASE r.rating
            WHEN 1.0 THEN -1.0
            WHEN 2.0 THEN -0.5
            WHEN 3.0 THEN  0.0
            WHEN 4.0 THEN  0.5
            WHEN 5.0 THEN  1.0
        END AS target
    FROM read_parquet('{INTERACTION_PATH}') r
    INNER JOIN read_parquet(
        '{OUTPUT_DIR / "mf_users.parquet"}'
    ) u
        ON r.user_id = u.user_id
    INNER JOIN read_parquet(
        '{OUTPUT_DIR / "mf_items.parquet"}'
    ) i
        ON r.parent_asin = i.parent_asin
    WHERE EXTRACT(
        YEAR FROM to_timestamp(r.timestamp / 1000)
    ) <= 2021
"""

In [17]:
DUCKDB_CHUNK_SIZE = 100_000

train_result = con.execute(TRAINING_QUERY)

chunk = train_result.fetch_df_chunk(
    DUCKDB_CHUNK_SIZE
)

print(chunk.shape)
print(chunk.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

: 